# 500% Forward-Return Stock Price-Structure Research

This notebook scans daily Parquet data under `/content/drive/MyDrive/quant/data/parquet/daily`, identifies stock/date observations where the stock reaches at least **+500% forward return within 252 trading days**, computes only **pre-event / information-available-at-the-time** price-structure features, and exports the results to Google Drive.

**Important:** A 500% forward return means the future maximum close within the next 252 trading sessions is at least 6× the current close. Features are calculated only from data up to the observation date, avoiding look-ahead leakage in the feature set.

In [ ]:
# 01 — Mount Google Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, glob, json, math, warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/content/drive/MyDrive/quant/data/parquet/daily')
OUTPUT_DIR = Path('/content/drive/MyDrive/quant/results/500pct_price_structure')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data directory:', DATA_DIR)
print('Output directory:', OUTPUT_DIR)
print('Data directory exists:', DATA_DIR.exists())
print('Parquet files:', len(list(DATA_DIR.rglob('*.parquet'))))

In [ ]:
# 02 — Imports and research configuration
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

FORWARD_DAYS = 252
WINNER_RETURN = 5.0       # +500% = ending/future price >= 6x current price
MIN_HISTORY = 252
        
WINDOWS = [20, 60, 120, 252]

# Set True if you want to keep only liquid enough observations later.
APPLY_LIQUIDITY_FILTER = False
MIN_MEDIAN_DAILY_TURNOVER_60D = 1_000_000

print('Forward horizon:', FORWARD_DAYS)
print('Winner threshold:', f'{WINNER_RETURN:.0%}')

## Expected input schema

The loader accepts common NSE/Parquet naming conventions. It tries to identify columns for date, symbol, open, high, low, close and volume. If your files use different names, edit the `COLUMN_ALIASES` cell below.


In [ ]:
# 03 — Flexible column mapping
COLUMN_ALIASES = {
    'date':   ['date', 'datetime', 'timestamp', 'trade_date', 'dt'],
    'symbol': ['symbol', 'ticker', 'tradingsymbol', 'security', 'stock', 'name'],
    'open':   ['open', 'open_price'],
    'high':   ['high', 'high_price'],
    'low':    ['low', 'low_price'],
    'close':  ['close', 'close_price', 'last', 'ltp'],
    'volume': ['volume', 'vol', 'total_traded_qty', 'quantity'],
}

def find_column(columns, aliases):
    normalized = {str(c).strip().lower(): c for c in columns}
    for alias in aliases:
        if alias in normalized:
            return normalized[alias]
    return None

def standardize_columns(df):
    mapping = {}
    for target, aliases in COLUMN_ALIASES.items():
        col = find_column(df.columns, aliases)
        if col is not None:
            mapping[col] = target
    df = df.rename(columns=mapping)
    required = ['date', 'symbol', 'open', 'high', 'low', 'close']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Found: {list(df.columns)}')
    if 'volume' not in df.columns:
        df['volume'] = np.nan
    return df[['date','symbol','open','high','low','close','volume']]


In [ ]:
# 04 — Inspect the Parquet collection before loading everything
files = sorted(DATA_DIR.rglob('*.parquet'))
if not files:
    raise FileNotFoundError(f'No parquet files found under {DATA_DIR}')

print(f'Found {len(files):,} parquet files')
for f in files[:10]:
    print(' ', f.relative_to(DATA_DIR))
if len(files) > 10:
    print(' ...')

sample = pd.read_parquet(files[0])
print('\nFirst file:', files[0].name)
print('Columns:', list(sample.columns))
print('Rows:', len(sample))
display(sample.head())

In [ ]:
# 05 — Load and normalize all daily data
def load_all_parquet(data_dir):
    parquet_files = sorted(Path(data_dir).rglob('*.parquet'))
    frames = []
    errors = []

    for i, path in enumerate(parquet_files, 1):
        try:
            df = pd.read_parquet(path)
            df = standardize_columns(df)
            frames.append(df)
        except Exception as e:
            errors.append((str(path), str(e)))
        if i % 100 == 0 or i == len(parquet_files):
            print(f'Loaded {i:,}/{len(parquet_files):,} files')

    if not frames:
        raise RuntimeError('No readable Parquet files.')

    data = pd.concat(frames, ignore_index=True)
    data['date'] = pd.to_datetime(data['date'], errors='coerce')
    for c in ['open','high','low','close','volume']:
        data[c] = pd.to_numeric(data[c], errors='coerce')
        
    data['symbol'] = data['symbol'].astype(str).str.strip().str.upper()
            data = data.dropna(subset=['date','symbol','close'])
    data = data[data['close'] > 0]
    data = data.sort_values(['symbol','date'])
    data = data.drop_duplicates(['symbol','date'], keep='last').reset_index(drop=True)
    return data, errors

data, load_errors = load_all_parquet(DATA_DIR)
print('\nRows:', f'{len(data):,}')
print('Symbols:', f"{data['symbol'].nunique():,}")
print('Date range:', data['date'].min().date(), 'to', data['date'].max().date())
print('Unreadable files:', len(load_errors))
if load_errors:
    print(load_errors[:5])

In [ ]:
# 06 — Optional data-quality summary
quality = data.groupby('symbol').agg(
    first_date=('date','min'),
    last_date=('date','max'),
    rows=('date','size'),
    missing_volume=('volume', lambda x: x.isna().mean())
).sort_values('rows', ascending=False)

display(quality.head(20))
print('Symbols with >= 252 observations:', (quality['rows'] >= MIN_HISTORY).sum())

In [ ]:
# 07 — Compute the forward 252-session maximum return
#
# For each symbol/date, future_max_close is the maximum CLOSE from the
# next 252 observations (excluding today's close). This makes the event
# label explicitly forward-looking while keeping all feature columns backward-looking.

def add_forward_labels(g):
    g = g.sort_values('date').copy()
    future_max = g['close'].shift(-1).rolling(FORWARD_DAYS, min_periods=FORWARD_DAYS).max().shift(-(FORWARD_DAYS-1))
    g['future_max_close_252d'] = future_max
    g['future_return_252d_max'] = future_max / g['close'] - 1.0
    g['winner_500pct_252d'] = g['future_return_252d_max'] >= WINNER_RETURN
    return g

data = data.groupby('symbol', group_keys=False).apply(add_forward_labels)
data = data.reset_index(drop=True)

winners = data[data['winner_500pct_252d']].copy()
print('500%+ observations:', f'{len(winners):,}')
print('Unique winner symbols:', winners['symbol'].nunique())
display(winners[['symbol','date','close','future_max_close_252d','future_return_252d_max']].head(20))

## Pre-breakout price-structure features

All features below use only historical observations through the current date. They are designed to capture compression, trend, momentum, breakout proximity, volatility, drawdown, volume behavior and price geometry.

In [ ]:
# 08 — Feature engineering
def add_features(g):
    g = g.sort_values('date').copy()
    c, h, l, v = g['close'], g['high'], g['low'], g['volume']

    # Returns / momentum
    for w in [5, 20, 60, 120, 252]:
        g[f'ret_{w}d'] = c / c.shift(w) - 1

    # Moving averages and trend alignment
    for w in [20, 50, 100, 200]:
        ma = c.rolling(w, min_periods=w).mean()
        g[f'sma_{w}'] = ma
        g[f'close_sma_{w}_ratio'] = c / ma - 1
        g[f'sma_{w}_slope_20d'] = ma / ma.shift(20) - 1

    g['sma_50_gt_100'] = (g['sma_50'] > g['sma_100']).astype('int8')
    g['sma_100_gt_200'] = (g['sma_100'] > g['sma_200']).astype('int8')
    g['sma_50_gt_200'] = (g['sma_50'] > g['sma_200']).astype('int8')

    # True range / ATR and volatility
    prev_close = c.shift(1)
    tr = pd.concat([(h-l), (h-prev_close).abs(), (l-prev_close).abs()], axis=1).max(axis=1)
    for w in [14, 20, 60]:
        atr = tr.rolling(w, min_periods=w).mean()
        g[f'atr_{w}'] = atr
        g[f'atr_{w}_pct'] = atr / c
    g['atr14_pct_change_20d'] = g['atr_14_pct'] / g['atr_14_pct'].shift(20) - 1

    logret = np.log(c / c.shift(1))
    for w in [20, 60, 120]:
        g[f'volatility_{w}d'] = logret.rolling(w, min_periods=w).std() * np.sqrt(252)

    # Rolling highs/lows; exclude today to measure distance to prior resistance
    for w in [20, 60, 120, 252]:
        prior_high = h.shift(1).rolling(w, min_periods=w).max()
        prior_low = l.shift(1).rolling(w, min_periods=w).min()
        g[f'prior_high_{w}d'] = prior_high
        g[f'prior_low_{w}d'] = prior_low
        g[f'distance_prior_high_{w}d'] = c / prior_high - 1
        g[f'distance_prior_low_{w}d'] = c / prior_low - 1
        g[f'range_{w}d_pct'] = (prior_high - prior_low) / c

    # Range compression: current 20/60-day range versus longer range
    g['range20_to_range120'] = g['range_20d_pct'] / g['range_120d_pct']
    g['range60_to_range252'] = g['range_60d_pct'] / g['range_252d_pct']

    # Position inside prior ranges
    for w in [60, 120, 252]:
        hi = g[f'prior_high_{w}d']
        lo = g[f'prior_low_{w}d']
        g[f'range_position_{w}d'] = (c - lo) / (hi - lo)

    # Drawdown from prior rolling high
    for w in [60, 120, 252]:
        hi = g[f'prior_high_{w}d']
        g[f'drawdown_from_high_{w}d'] = c / hi - 1

    # Recent consolidation / tightness
    daily_range_pct = (h-l) / c.replace(0, np.nan)
    g['avg_daily_range_20d'] = daily_range_pct.rolling(20, min_periods=20).mean()
    g['avg_daily_range_60d'] = daily_range_pct.rolling(60, min_periods=60).mean()
    g['range_tightening_20_vs_60'] = g['avg_daily_range_20d'] / g['avg_daily_range_60d']

    # Count recent closes near prior 60/252-day high
    prior_60_high = g['prior_high_60d']
    prior_252_high = g['prior_high_252d']
    g['days_near_60d_high_20d'] = ((c >= prior_60_high * 0.95).astype(int).rolling(20, min_periods=20).sum())
    g['days_near_252d_high_60d'] = ((c >= prior_252_high * 0.90).astype(int).rolling(60, min_periods=60).sum())

    # Higher-low / higher-high structure
    g['higher_low_20_vs_60'] = (g['prior_low_20d'] > g['prior_low_60d']).astype('int8')
    g['higher_high_20_vs_60'] = (g['prior_high_20d'] > g['prior_high_60d']).astype('int8')

    # Volume behavior
    if v.notna().any():
        v20 = v.rolling(20, min_periods=10).mean()
        v60 = v.rolling(60, min_periods=20).mean()
        g['volume_vs_20d'] = v / v20
        g['volume_vs_60d'] = v / v60
        g['volume_trend_20_vs_60'] = v20 / v60 - 1
        g['volume_expansion_5_vs_60'] = v.rolling(5, min_periods=3).mean() / v60
        g['turnover'] = c * v
        g['median_turnover_60d'] = g['turnover'].rolling(60, min_periods=20).median()
    else:
        for col in ['volume_vs_20d','volume_vs_60d','volume_trend_20_vs_60','volume_expansion_5_vs_60','turnover','median_turnover_60d']:
            g[col] = np.nan

    # Candle / breakout geometry
    g['body_pct'] = (c - g['open']) / c.replace(0, np.nan)
    g['close_location_in_day'] = (c - l) / (h - l).replace(0, np.nan)
    g['upper_wick_pct'] = (h - np.maximum(g['open'], c)) / c
    g['lower_wick_pct'] = (np.minimum(g['open'], c) - l) / c

    # Distance to prior 52W high in ATR units
    g['distance_252d_high_atr'] = (g['prior_high_252d'] - c) / g['atr_14'].replace(0, np.nan)

    # Acceleration: recent momentum minus prior momentum
    g['momentum_acceleration'] = g['ret_20d'] - g['ret_20d'].shift(20)

    return g

data = data.groupby('symbol', group_keys=False).apply(add_features)
data = data.reset_index(drop=True)
print('Feature columns:', len(data.columns))

In [ ]:
# 09 — Build the event dataset
# Keep observations with enough history and a known forward outcome.
feature_cols = [c for c in data.columns if c not in {
    'date','symbol','future_max_close_252d','future_return_252d_max','winner_500pct_252d'
}]

events = data.dropna(subset=['future_max_close_252d']).copy()
events = events[events.groupby('symbol').cumcount() >= MIN_HISTORY].copy()

if APPLY_LIQUIDITY_FILTER:
    events = events[events['median_turnover_60d'] >= MIN_MEDIAN_DAILY_TURNOVER_60D]

winner_events = events[events['winner_500pct_252d']].copy()
nonwinner_events = events[~events['winner_500pct_252d']].copy()

print('Event observations:', f'{len(events):,}')
print('500%+ winner observations:', f'{len(winner_events):,}')
print('Unique winner symbols:', winner_events['symbol'].nunique())
print('Unique symbols in sample:', events['symbol'].nunique())

In [ ]:
# 10 — Identify the first qualifying 500% event per stock / run
# A stock can have multiple qualifying dates during one large run. To avoid
# counting every day of the same run as an independent discovery, we retain
# the earliest qualifying observation after a non-winning observation.
winner_events = winner_events.sort_values(['symbol','date']).copy()
winner_events['new_winner_run'] = (
    (~winner_events['winner_500pct_252d'].shift(1).fillna(False)) |
    (winner_events['symbol'] != winner_events['symbol'].shift(1))
)

# Since winner_events itself contains only winners, the above is insufficient
# to detect a gap in the full event series. Use the full data for a clean run id.
events = events.sort_values(['symbol','date']).copy()
prev_win = events.groupby('symbol')['winner_500pct_252d'].shift(1).fillna(False)
events['winner_run_start'] = events['winner_500pct_252d'] & ~prev_win

first_events = events[events['winner_run_start']].copy()
print('Independent winner runs:', len(first_events))
display(first_events[['symbol','date','close','future_max_close_252d','future_return_252d_max']].sort_values('future_return_252d_max', ascending=False).head(30))

In [ ]:
# 11 — Winner-level dataset
# Include the pre-event feature state plus a compact set of outcome columns.
outcome_cols = [
    'symbol','date','close','future_max_close_252d','future_return_252d_max',
    'winner_500pct_252d'
]
winner_dataset = first_events[outcome_cols + feature_cols].copy()

# Keep a clean, useful ordering.
priority = [
    'symbol','date','close','future_max_close_252d','future_return_252d_max',
    'ret_20d','ret_60d','ret_120d','ret_252d',
    'close_sma_50_ratio','close_sma_100_ratio','close_sma_200_ratio',
    'sma_50_slope_20d','sma_100_slope_20d','sma_200_slope_20d',
    'range_60d_pct','range_120d_pct','range_252d_pct',
    'range20_to_range120','range60_to_range252',
    'range_position_60d','range_position_120d','range_position_252d',
    'distance_prior_high_60d','distance_prior_high_120d','distance_prior_high_252d',
    'atr_14_pct','atr14_pct_change_20d',
    'volatility_20d','volatility_60d','volatility_120d',
    'volume_vs_20d','volume_vs_60d','volume_trend_20_vs_60',
    'days_near_252d_high_60d','higher_low_20_vs_60','higher_high_20_vs_60',
    'distance_252d_high_atr'
]
priority = [c for c in priority if c in winner_dataset.columns]
remaining = [c for c in winner_dataset.columns if c not in priority]
winner_dataset = winner_dataset[priority + remaining]

display(winner_dataset.head())

## Winner vs non-winner comparison

The winner-only dataset is useful for discovering the shape of extreme winners. The comparison below is important for determining which features are actually unusual rather than merely common across all stocks.

In [ ]:
# 12 — Feature distribution comparison
comparison_features = [
    'ret_20d','ret_60d','ret_120d','ret_252d',
    'close_sma_50_ratio','close_sma_100_ratio','close_sma_200_ratio',
    'sma_50_slope_20d','sma_100_slope_20d','sma_200_slope_20d',
    'range_60d_pct','range_120d_pct','range_252d_pct',
    'range20_to_range120','range60_to_range252',
    'range_position_60d','range_position_120d','range_position_252d',
    'distance_prior_high_60d','distance_prior_high_120d','distance_prior_high_252d',
    'atr_14_pct','atr14_pct_change_20d',
    'volatility_20d','volatility_60d',
    'volume_vs_20d','volume_vs_60d','volume_trend_20_vs_60',
    'days_near_252d_high_60d','distance_252d_high_atr'
]
comparison_features = [c for c in comparison_features if c in events.columns]

rows = []
for f in comparison_features:
    w = events.loc[events['winner_500pct_252d'], f].dropna()
    n = events.loc[~events['winner_500pct_252d'], f].dropna()
    if len(w) < 10 or len(n) < 10:
        continue
    rows.append({
        'feature': f,
        'winner_n': len(w),
        'nonwinner_n': len(n),
        'winner_median': w.median(),
        'nonwinner_median': n.median(),
        'winner_mean': w.mean(),
        'nonwinner_mean': n.mean(),
        'median_difference': w.median() - n.median(),
        'median_ratio': w.median() / n.median() if n.median() != 0 else np.nan,
    })

comparison = pd.DataFrame(rows).sort_values('median_difference', key=lambda x: x.abs(), ascending=False)
display(comparison.head(30))

In [ ]:
# 13 — Percentile profile of 500% winners
profile_features = [c for c in comparison_features if c in winner_dataset.columns]
profile = winner_dataset[profile_features].describe(percentiles=[.10,.25,.50,.75,.90]).T
profile = profile.rename(columns={'50%':'median','25%':'p25','75%':'p75','10%':'p10','90%':'p90'})
display(profile[['count','mean','p10','p25','median','p75','p90']].sort_values('median'))

In [ ]:
# 14 — Save results to Google Drive
winner_path = OUTPUT_DIR / 'winner_500pct_events.csv'
winner_parquet = OUTPUT_DIR / 'winner_500pct_events.parquet'
all_events_path = OUTPUT_DIR / 'all_event_observations.csv'
comparison_path = OUTPUT_DIR / 'winner_vs_nonwinner_feature_comparison.csv'
profile_path = OUTPUT_DIR / 'winner_feature_profile.csv'
quality_path = OUTPUT_DIR / 'data_quality_by_symbol.csv'
errors_path = OUTPUT_DIR / 'load_errors.csv'

winner_dataset.to_csv(winner_path, index=False)
winner_dataset.to_parquet(winner_parquet, index=False)
events.to_csv(all_events_path, index=False)
comparison.to_csv(comparison_path, index=False)
profile.to_csv(profile_path)
quality.to_csv(quality_path)
pd.DataFrame(load_errors, columns=['file','error']).to_csv(errors_path, index=False)

print('Exported:')
for p in [winner_path,winner_parquet,all_events_path,comparison_path,profile_path,quality_path,errors_path]:
    print(' ', p, f'({p.stat().st_size/1024/1024:.2f} MB)')

In [ ]:
# 15 — Final research summary
summary = {
    'data_directory': str(DATA_DIR),
    'parquet_files': len(files),
    'rows': int(len(data)),
    'symbols': int(data['symbol'].nunique()),
    'date_start': str(data['date'].min().date()),
    'date_end': str(data['date'].max().date()),
    'event_observations': int(len(events)),
    'winner_observations': int(len(winner_events)),
    'independent_winner_runs': int(len(first_events)),
    'unique_winner_symbols': int(first_events['symbol'].nunique()),
    'forward_horizon_trading_days': FORWARD_DAYS,
    'winner_threshold': WINNER_RETURN,
    'output_directory': str(OUTPUT_DIR),
}

print(json.dumps(summary, indent=2))
with open(OUTPUT_DIR / 'research_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\nTop 20 500% winner events by forward maximum return:')
display(first_events[['symbol','date','close','future_max_close_252d','future_return_252d_max']]
        .sort_values('future_return_252d_max', ascending=False).head(20))